In [1]:
import re
import pandas as pd
from huggingface_hub import list_repo_files

files = list_repo_files("edcci/GenECG", repo_type="dataset")
image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

records = []
for f in image_files:
    match = re.search(r'(\d+)_hr_1R\.png$', f)
    if match:
        ecg_id = int(match.group(1))
        dataset_variant = "A" if "Dataset_A" in f else "B"
        records.append({"path": f, "ecg_id": ecg_id, "variant": dataset_variant})

genecg_df = pd.DataFrame(records)

genecg_df.to_csv("genecg_mapping.csv", index=False)

c:\Users\neifa\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
genecg_df = pd.read_csv("genecg_mapping.csv")  # on recharge depuis le disque
ptbxl_df = pd.read_csv(r"C:\Users\neifa\Documents\ENETCOM\3IDSD\stage_25-26\cardiac-multiagent-pipeline\physionet\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\ptbxl_database.csv")

# Pivot : une ligne par ecg_id, avec une colonne par variante
pivoted = genecg_df.pivot(index="ecg_id", columns="variant", values="path")
pivoted.columns = [f"path_{col}" for col in pivoted.columns]
pivoted = pivoted.reset_index()

# Merge avec les métadonnées PTB-XL
merged = ptbxl_df.merge(pivoted, on="ecg_id", how="left")
merged.to_csv("ptbxl_with_genecg_paths.csv", index=False)  # le fichier final

print(merged.shape)
print(merged[["ecg_id", "path_A", "path_B"]].head())

(21799, 30)
   ecg_id                                             path_A  \
0       1  Dataset_A_ECGs_without_imperfections/00000/000...   
1       2  Dataset_A_ECGs_without_imperfections/00000/000...   
2       3  Dataset_A_ECGs_without_imperfections/00000/000...   
3       4  Dataset_A_ECGs_without_imperfections/00000/000...   
4       5  Dataset_A_ECGs_without_imperfections/00000/000...   

                                              path_B  
0  Dataset_B_ECGs_with_imperfections/00000/00001_...  
1  Dataset_B_ECGs_with_imperfections/00000/00002_...  
2  Dataset_B_ECGs_with_imperfections/00000/00003_...  
3  Dataset_B_ECGs_with_imperfections/00000/00004_...  
4  Dataset_B_ECGs_with_imperfections/00000/00005_...  


In [3]:
genecg_df = pd.read_csv("genecg_mapping.csv")